# Bird Identification using CV - Part 7: Video with classification

**ITAI 1378  |  Midterm  |  2026**

**Group 8**

**Author:** Stuart Fairchild | Kalen Foster | Ranveer Chand

---

In [1]:
%pip install -q ultralytics

from ultralytics import YOLO, SAM
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

print("Setup complete. Ready to detect and segment.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete. Ready to detect and segment.


---

## Load images from Google Drive

Connect to Google Drive to load test image directory.

To use this without changes, add images to Google Drive under

`/Colab Notebooks/ITAI1378/midterm/images`

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import cv2
base_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm'
videos_path = f"{base_path}/videos"

for filename in os.listdir(videos_path):
  if filename.endswith((".mp4")):
    file_path = os.path.join(videos_path, filename)
    print(f"Current file: {filename}")

Current file: cut.mp4


## Running detection with YOLO11 Large on video


In [6]:
import os
import cv2
from ultralytics import YOLO
# Import Annotator for drawing bounding boxes and labels on frames
from ultralytics.utils.plotting import Annotator
import numpy as np
# import matplotlib.pyplot as plt # Removed matplotlib import

base_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm'
videos_path = f"{base_path}/videos"

# --- Main Bird Detector (COCO 'bird' class) ---
model_name_general = "YOLO11Large_videoWithGPU_GeneralBird"
general_bird_detector = YOLO("yolo11l.pt")
print(f"{model_name_general} loaded.")

# --- Blue Jay Classifier (custom model) ---
multibird_model_path = "/content/models/multi-bird.pt"
# Assuming multi-bird.pt is a detection model trained on specific bird classes.
multibird_classifier = YOLO(multibird_model_path)
print("Multi-bird classifier loaded.")
# DEBUG: Print the class names of the multi-bird classifier
print(f"Multi-bird classifier class names: {multibird_classifier.names}")

# Output path for the processed videos
processed_videos_path = f"{base_path}/detected_and_classified_videos"
os.makedirs(processed_videos_path, exist_ok=True)
print(f"Output videos will be saved to: {processed_videos_path}")

# Output path for debugging cropped bird images
debug_crops_path = f"{base_path}/debug_bird_crops"
os.makedirs(debug_crops_path, exist_ok=True)
print(f"Debug cropped images will be saved to: {debug_crops_path}")

for filename in os.listdir(videos_path):
  if filename.endswith((".mp4")):
    file_path = os.path.join(videos_path, filename)
    print(f"\nProcessing video: {filename}")

    cap = cv2.VideoCapture(file_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {filename}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # Use MP4V codec for broader compatibility.
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    output_video_file = os.path.join(processed_videos_path, f"cls_{filename}")
    out = cv2.VideoWriter(output_video_file, fourcc, fps, (width, height))

    frame_idx = 0
    seen_bird_ids = set() # Set to store unique track IDs for counting total birds

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run general bird detection AND tracking on the current frame
        # Using 'track' instead of 'predict' to get object IDs across frames
        general_results = general_bird_detector.track(
            source=frame,
            conf=0.12,           # Set confidence threshold lower
            classes=[14],        # COCO index for 'bird'
            save=False,          # We will handle saving manually
            verbose=False,       # Suppress per-frame output
            tracker="bytetrack.yaml" # Use a tracker configuration
        )

        # Create a copy of the frame for drawing annotations
        annotated_frame = frame.copy()
        annotator = Annotator(annotated_frame, line_width=2) # Initialize annotator for this frame

        # Process results from the general bird detector (list of Results objects, typically one per image source)
        for r in general_results:
            if r.boxes is not None and r.boxes.id is not None: # Check if tracking IDs are available
                for box, track_id in zip(r.boxes, r.boxes.id):
                    # Extract bounding box coordinates and confidence for the general bird detection
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                    bird_conf = box.conf[0].cpu().numpy()
                    track_id = int(track_id.cpu().numpy()) # Convert track_id to integer

                    seen_bird_ids.add(track_id) # Add unique track ID to the set

                    # Ensure coordinates are within image bounds before cropping
                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(width, x2)
                    y2 = min(height, y2)

                    cropped_bird_img = frame[y1:y2, x1:x2]

                    # Initialize label and color for the general bird detection
                    final_label = f"Bird {track_id} {bird_conf:.2f}" # Added track_id to label
                    final_draw_color = (255, 255, 0) # BGR Cyan for general birds

                    # Only attempt multi-bird detection if a valid crop exists
                    if cropped_bird_img.shape[0] > 0 and cropped_bird_img.shape[1] > 0:

                        # Run multi-bird detection on the cropped bird image
                        multi_bird_results = multibird_classifier.predict(
                            source=cropped_bird_img,
                            verbose=False, # Suppress per-frame output for this model
                            save=False     # Do not save intermediate detection images
                        )

                        # Process results from the multi-bird model (which is a detection model)
                        if multi_bird_results and len(multi_bird_results) > 0:
                            mb_r = multi_bird_results[0] # Get the single result for the cropped image

                            if mb_r.boxes is not None and len(mb_r.boxes) > 0: # Check for detections within the cropped image
                                # Iterate through detections found by the multi-bird model within the cropped bird image
                                for sub_box in mb_r.boxes:
                                    sub_class_idx = int(sub_box.cls[0].cpu().numpy())
                                    sub_class_conf = sub_box.conf[0].cpu().numpy()
                                    predicted_class_name = multibird_classifier.names[sub_class_idx]

                                    # DEBUG: Print classification result for this bird, including general bird confidence
                                    print(f"  Frame {frame_idx} (Track {track_id}): Sub-classified bird (from general bbox [{x1},{y1},{x2},{y2}]) as '{predicted_class_name}' with conf {sub_class_conf:.2f} (General Bird conf: {bird_conf:.2f})")

                                    # Check if the detected sub-class is a 'Blue Jay' with sufficient confidence
                                    # Lowered threshold to 0.2 to make re-labeling more likely for debugging
                                    if 'american robin' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"American Robin {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (128, 0, 255) # BGR
                                        break
                                    if 'blue jay' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"Blue Jay {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (255, 0, 0) # Blue for 'Blue Jay'
                                        break
                                    if 'downy woodpecker' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"Downy Woodpecker {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (225, 200, 255) # Light magenta for 'Downy Woodpecker'
                                        break
                                    if 'house finch' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"House Finch {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (100, 200, 255)
                                        break
                                    if 'mourning dove' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"Mourning Dove {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (128, 128, 128) # Gray for 'Mourning Dove'
                                        break
                                    if 'northern cardinal' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"Northern Cardinal {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (0, 0, 255) # Red
                                        break
                                    if 'red-bellied woodpecker' in predicted_class_name.lower() and sub_class_conf > 0.45:
                                        final_label = f"Red-bellied Woodpecker {track_id} {sub_class_conf:.2f}"
                                        final_draw_color = (128, 128, 255) # Light red for 'Red-bellied Woodpecker'
                                        break

                    # DEBUG: Print the final label and color that will be drawn
                    print(f"  Frame {frame_idx} (Track {track_id}): Drawing label '{final_label}' with color {final_draw_color} for bbox [{x1},{y1},{x2},{y2}]")

                    # Draw the bounding box and label on the annotated_frame
                    annotator.box_label([x1, y1, x2, y2], final_label, color=final_draw_color)

        out.write(annotator.result()) # Write the fully annotated frame to the output video

        frame_idx += 1
        if frame_idx % 100 == 0:
            print(f"  Processed {frame_idx} frames for {filename}")

    cap.release()
    out.release()
    print(f"Finished processing {filename}. Output saved to {output_video_file}")
    print(f"Total unique birds tracked in {filename}: {len(seen_bird_ids)}")

print("All videos processed.")

Streaming output truncated to the last 5000 lines.
  Frame 216: Drawing label 'Red-bellied Woodpecker 0.88' with color (128, 128, 255) for bbox [1475,1409,1645,1649]
  Frame 217: Sub-classified bird (from general bbox [1469,1415,1649,1647]) as 'Red-bellied Woodpecker' with conf 0.93 (General Bird conf: 0.33)
  Frame 217: Drawing label 'Red-bellied Woodpecker 0.93' with color (128, 128, 255) for bbox [1469,1415,1649,1647]
  Frame 217: Sub-classified bird (from general bbox [2125,348,2330,566]) as 'Mourning Dove' with conf 0.52 (General Bird conf: 0.20)
  Frame 217: Drawing label 'Mourning Dove 0.52' with color (128, 128, 128) for bbox [2125,348,2330,566]
  Frame 217: Sub-classified bird (from general bbox [1471,1412,1735,1648]) as 'Red-bellied Woodpecker' with conf 0.93 (General Bird conf: 0.17)
  Frame 217: Drawing label 'Red-bellied Woodpecker 0.93' with color (128, 128, 255) for bbox [1471,1412,1735,1648]
  Frame 218: Sub-classified bird (from general bbox [2126,350,2331,566]) as 'Ho

In [8]:
import os
import cv2
from ultralytics import YOLO
# Import Annotator for drawing bounding boxes and labels on frames
from ultralytics.utils.plotting import Annotator
import numpy as np
import json # Added for JSON output

base_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm'
videos_path = f"{base_path}/videos"

# --- Main Bird Detector (COCO 'bird' class) ---
model_name_general = "YOLO11Large_videoWithGPU_GeneralBird"
general_bird_detector = YOLO("yolo11l.pt")
print(f"{model_name_general} loaded.")

# --- Multi-Bird Classifier (custom detection model) ---
multibird_model_path = "/content/models/multi-bird.pt"
multibird_classifier = YOLO(multibird_model_path)
print("Multi-bird classifier loaded.")
print(f"Multi-bird classifier class names: {multibird_classifier.names}")

# Output paths
processed_videos_path = f"{base_path}/detected_and_classified_videos"
os.makedirs(processed_videos_path, exist_ok=True)
print(f"Output videos will be saved to: {processed_videos_path}")

bird_summary_output_path = f"{base_path}/bird_classification_summaries"
os.makedirs(bird_summary_output_path, exist_ok=True)
os.makedirs(os.path.join(bird_summary_output_path, 'images'), exist_ok=True) # Subdirectory for images
print(f"Bird classification summaries (JSON and images) will be saved to: {bird_summary_output_path}")

# Removed debug_crops_path as its functionality is replaced by bird_summary_output_path/images

for filename in os.listdir(videos_path):
  if filename.endswith((".mp4")):
    file_path = os.path.join(videos_path, filename)
    print(f"\nProcessing video: {filename}")

    cap = cv2.VideoCapture(file_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {filename}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    output_video_file = os.path.join(processed_videos_path, f"cls_{filename}")
    out = cv2.VideoWriter(output_video_file, fourcc, fps, (width, height))

    frame_idx = 0
    seen_bird_ids = set() # Set to store unique track IDs for counting total birds
    bird_class_summary = {} # Stores best classification and image for each track_id throughout the video

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        general_results = general_bird_detector.track(
            source=frame,
            conf=0.12,
            classes=[14],
            save=False,
            verbose=False,
            tracker="bytetrack.yaml"
        )

        annotated_frame = frame.copy()
        annotator = Annotator(annotated_frame, line_width=2)

        for r in general_results:
            if r.boxes is not None and r.boxes.id is not None:
                for box, track_id in zip(r.boxes, r.boxes.id):
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                    bird_conf = box.conf[0].cpu().numpy()
                    track_id = int(track_id.cpu().numpy())

                    seen_bird_ids.add(track_id)

                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(width, x2)
                    y2 = min(height, y2)

                    cropped_bird_img = frame[y1:y2, x1:x2]

                    # Initialize label and color for the general bird detection
                    final_label = f"Bird {track_id} {bird_conf:.2f}"
                    final_draw_color = (255, 255, 0) # BGR Cyan for general birds

                    # Only attempt multi-bird detection if a valid crop exists
                    if cropped_bird_img.shape[0] > 0 and cropped_bird_img.shape[1] > 0:

                        multi_bird_results = multibird_classifier.predict(
                            source=cropped_bird_img,
                            verbose=False,
                            save=False
                        )

                        if multi_bird_results and len(multi_bird_results) > 0:
                            mb_r = multi_bird_results[0]

                            if mb_r.boxes is not None and len(mb_r.boxes) > 0:
                                # Variables to hold the best classification for *this specific general bird detection in the current frame*
                                best_sub_class_for_frame = None
                                highest_sub_conf_for_frame = -1.0

                                for sub_box in mb_r.boxes:
                                    sub_class_idx = int(sub_box.cls[0].cpu().numpy())
                                    sub_class_conf = sub_box.conf[0].cpu().numpy()
                                    predicted_class_name = multibird_classifier.names[sub_class_idx]

                                    print(f"  Frame {frame_idx} (Track {track_id}): Sub-classified bird (from general bbox [{x1},{y1},{x2},{y2}]) as '{predicted_class_name}' with conf {sub_class_conf:.2f} (General Bird conf: {bird_conf:.2f})")

                                    # Update overall best classification for this track_id (across all frames)
                                    if track_id not in bird_class_summary or \
                                       sub_class_conf > bird_class_summary[track_id]['best_confidence']:
                                        bird_class_summary[track_id] = {
                                            'best_class': predicted_class_name,
                                            'best_confidence': sub_class_conf,
                                            'representative_image': cropped_bird_img.copy() # Store image with best confidence
                                        }

                                    # Determine the best sub-classification for the *current frame's display*
                                    # This logic ensures that if multiple sub-detections are made, the one with highest conf is used for display.
                                    if sub_class_conf > highest_sub_conf_for_frame:
                                        highest_sub_conf_for_frame = sub_class_conf
                                        best_sub_class_for_frame = predicted_class_name

                                # After iterating all sub_boxes, set final_label and final_draw_color based on the best for this frame
                                if best_sub_class_for_frame is not None and highest_sub_conf_for_frame > 0.45: # Apply display threshold
                                    final_label = f"{best_sub_class_for_frame} {track_id} {highest_sub_conf_for_frame:.2f}" # Use sub_class_conf for display
                                    if 'american robin' in best_sub_class_for_frame.lower():
                                        final_draw_color = (128, 0, 255) # BGR Purple
                                    elif 'blue jay' in best_sub_class_for_frame.lower():
                                        final_draw_color = (255, 0, 0) # BGR Blue
                                    elif 'downy woodpecker' in best_sub_class_for_frame.lower():
                                        final_draw_color = (225, 200, 255) # BGR Light magenta
                                    elif 'house finch' in best_sub_class_for_frame.lower():
                                        final_draw_color = (100, 200, 255) # BGR Orange
                                    elif 'mourning dove' in best_sub_class_for_frame.lower():
                                        final_draw_color = (128, 128, 128) # BGR Gray
                                    elif 'northern cardinal' in best_sub_class_for_frame.lower():
                                        final_draw_color = (0, 0, 255) # BGR Red
                                    elif 'red-bellied woodpecker' in best_sub_class_for_frame.lower():
                                        final_draw_color = (128, 128, 255) # BGR Light red

                    print(f"  Frame {frame_idx} (Track {track_id}): Drawing label '{final_label}' with color {final_draw_color} for bbox [{x1},{y1},{x2},{y2}]")

                    # Draw the bounding box and label on the annotated_frame
                    annotator.box_label([x1, y1, x2, y2], final_label, color=final_draw_color)

        out.write(annotator.result()) # Write the fully annotated frame to the output video

        frame_idx += 1
        if frame_idx % 100 == 0:
            print(f"  Processed {frame_idx} frames for {filename}")

    cap.release()
    out.release()
    print(f"Finished processing {filename}. Output saved to {output_video_file}")
    print(f"Total unique birds tracked in {filename}: {len(seen_bird_ids)}")

    # --- Generate JSON summary and save representative images for this video ---+
    video_summary_json_data = []
    print(f"Generating classification summary for {filename}...")
    for track_id, data in bird_class_summary.items():
        best_class_name = data['best_class']
        best_confidence = data['best_confidence']
        representative_image = data['representative_image']

        # Ensure filename is unique and descriptive
        img_filename = f"{os.path.splitext(filename)[0]}_track_{track_id}_{best_class_name.replace(' ', '_')}_{best_confidence:.2f}.jpg"
        img_filepath = os.path.join(bird_summary_output_path, 'images', img_filename)
        cv2.imwrite(img_filepath, representative_image)

        video_summary_json_data.append({
            'track_id': track_id,
            'best_predicted_class': best_class_name,
            'best_confidence': float(best_confidence), # Convert numpy float to Python float for JSON serialization
            'representative_image_filename': os.path.join('images', img_filename) # Path relative to the summary folder
        })

    json_filename = f"{os.path.splitext(filename)[0]}_classification_summary.json"
    json_filepath = os.path.join(bird_summary_output_path, json_filename)
    with open(json_filepath, 'w') as f:
        json.dump(video_summary_json_data, f, indent=4)
    print(f"Classification summary saved to {json_filepath}")
    print(f"Representative images saved to {os.path.join(bird_summary_output_path, 'images')}")


print("All videos processed.")

YOLO11Large_videoWithGPU_GeneralBird loaded.
Multi-bird classifier loaded.
Multi-bird classifier class names: {0: 'American Robin', 1: 'Blue Jay', 2: 'Downy Woodpecker', 3: 'House Finch', 4: 'Mourning Dove', 5: 'Northern Cardinal', 6: 'Red-bellied Woodpecker'}
Output videos will be saved to: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/detected_and_classified_videos
Bird classification summaries (JSON and images) will be saved to: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/bird_classification_summaries

Processing video: 20260726_152147474958.mp4
  Frame 0 (Track 1): Sub-classified bird (from general bbox [1590,585,1935,843]) as 'Blue Jay' with conf 0.98 (General Bird conf: 0.79)
  Frame 0 (Track 1): Drawing label 'Blue Jay 1 0.98' with color (255, 0, 0) for bbox [1590,585,1935,843]
  Frame 1 (Track 1): Drawing label 'Bird 1 0.74' with color (255, 255, 0) for bbox [1599,584,1934,841]
  Frame 2 (Track 1): Sub-classified bird (from general bbox [